<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Gradio_Playground_Gemini_AI_Residency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradio Playground: Building AI Apps and Chatbots with the Gemini API

**Decoding Data Science · AI Residency and Training Cohorts**
Instructor: Mohammad Arshad · Updated October 2026 · Gradio 6 · Google Gen AI SDK (`google-genai`)

---

This notebook is a hands-on reference for building user interfaces for AI applications with **Gradio**. Every example uses the **Gemini API**, with the key stored safely in **Colab Secrets**.

Work through it in order. Each section introduces one idea, explains it, then gives you a runnable app you can modify.

| Part | What you build | Gradio concept |
|---|---|---|
| 0 | Setup, API key in Colab Secrets, helper functions | Environment |
| 1 | Hello Gradio, AI summariser, structured extractor, vision app | `gr.Interface` |
| 2 | Eight chatbots: basic, streaming, persona, multimodal, web-grounded, tool-calling agent, thinking, chat-with-your-document | `gr.ChatInterface` |
| 3 | Model arena, custom chatbot with feedback, data analyst app | `gr.Blocks` |
| 4 | Text-to-speech, voice assistant, image generation and editing | Media components |
| 5 | The DDS AI Playground: everything in one tabbed app | `gr.Blocks` + `gr.Sidebar` + `gr.Tabs` |
| 6 | Sharing, deploying to Hugging Face Spaces, MCP server, calling your app from Python | Deployment |
| 7 | Cheat sheet, troubleshooting, resident exercises | Reference |

**How to use this notebook**
1. Run Part 0 first, every time you open the notebook. It sets up the API key, the Gemini client and the helper functions that every app uses.
2. After that, each app is self-contained. Run any cell that ends in `.launch()` and the app appears inside the notebook.
3. Change the code, re-run the cell, and the app updates. That is the whole learning loop.

## Part 0 · Setup

### 0.1 Add your Gemini API key to Colab Secrets

Never paste an API key directly into a notebook. Anyone you share the notebook with, or anyone who sees your screen, can use it, and keys pushed to GitHub are found by bots within minutes.

1. Create a key at **Google AI Studio**: https://aistudio.google.com/apikey
2. In Colab, click the **key icon** in the left sidebar (Secrets).
3. Click **Add new secret**. Name: `GEMINI_API_KEY`. Value: your key.
4. Switch on **Notebook access** for this notebook.

The code below reads the key from Colab Secrets. When you later deploy to Hugging Face Spaces, the same code reads it from the Space's Secrets (an environment variable), so you never change a line.

### 0.2 Install the libraries

Gradio is pre-installed in Colab, but we upgrade it so everyone in the cohort is on the same version. `google-genai` is Google's current official Python SDK for Gemini (it replaces the older `google-generativeai` package).

In [1]:
!pip install -q -U gradio google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [2]:
import os, io, json, time, mimetypes
import numpy as np
import pandas as pd
from PIL import Image

import gradio as gr
from google import genai
from google.genai import types

print("Gradio version:", gr.__version__)
print("google-genai version:", genai.__version__)

Gradio version: 6.30.0
google-genai version: 2.33.0


### 0.3 Connect to Gemini

This cell works in three places without changes:
- **Colab**: reads `GEMINI_API_KEY` from Colab Secrets.
- **Hugging Face Spaces / your laptop**: reads the `GEMINI_API_KEY` environment variable.
- **Anywhere else**: asks you to paste the key (hidden input, never saved).

In [3]:
def load_gemini_key():
    try:
        from google.colab import userdata          # only exists inside Colab
        key = userdata.get("gemini1")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    from getpass import getpass
    return getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=load_gemini_key())

# Model names used throughout the notebook (October 2026).
# If Google renames a model, change it here once and every app picks it up.
MODEL       = "gemini-3.8-flash"        # main model: smart and fast
FAST_MODEL  = "gemini-3.5-flash-lite"   # cheapest, for high-volume simple tasks
IMAGE_MODEL = "gemini-nano-banana-2.1"  # image generation and editing
TTS_MODEL   = "gemini-3.8-flash-tts"    # text-to-speech

response = client.models.generate_content(
    model=MODEL,
    contents="In one sentence, welcome a new AI Residency cohort to Gradio.",
)
print(response.text)

Welcome to the Gradio family, AI Residents—we can’t wait to see how you push the boundaries of machine learning and turn cutting-edge models into interactive, impactful applications!


**If the test call fails** with "model not found", run the cell below. It lists the models your key can use, so you can update the four model names above.

In [4]:
for m in client.models.list():
    name = m.name.replace("models/", "")
    if "gemini" in name:
        print(name)

gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3-pro-image-preview
gemini-3-pro-image
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-nano-banana-2.1
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-4-argon
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-robotics-er-2-preview
gemini-2.5-computer-use-preview-10-2025
gemini-embedding-001
gemini-embedding-2-preview
gemini-embedding-2
gemini-3.5-transcribe-live
gemini-2.5-flash-native-audio-latest
gemini-2.5-flash-native-audio-preview-09-2025
gemini-2.5-flash-native

### 0.4 Helper functions (run this, read it later)

Gradio and Gemini describe a conversation in slightly different ways:

| | Gradio chat history | Gemini `contents` |
|---|---|---|
| Roles | `"user"`, `"assistant"` | `"user"`, `"model"` |
| A message | `{"role": ..., "content": ...}` | `types.Content(role=..., parts=[...])` |
| Text | a string, or `{"type": "text", "text": ...}` | `types.Part.from_text(text=...)` |
| Files | `{"type": "file", "file": {"path": ...}}` | `types.Part.from_bytes(data=..., mime_type=...)` |

The helpers below translate between the two, so each chatbot in Part 2 stays short and readable. They also skip the collapsible "Thinking" and "Tool used" bubbles we show in some apps, because those are for the user to see, not for the model to re-read.

In [5]:
def file_to_part(path):
    """Turn a local file (image, PDF, audio, text...) into a Gemini Part."""
    mime = mimetypes.guess_type(path)[0] or "application/octet-stream"
    if mime.startswith("text/") or path.endswith((".md", ".csv", ".json", ".py")):
        mime = "text/plain"
    with open(path, "rb") as f:
        return types.Part.from_bytes(data=f.read(), mime_type=mime)


def _path_of(item):
    if isinstance(item, str):
        return item
    if isinstance(item, dict):
        f = item.get("file", item)
        return f.get("path") if isinstance(f, dict) else f
    return getattr(item, "path", None)


def content_to_parts(content):
    """Convert one Gradio message 'content' (any format) into Gemini Parts."""
    if isinstance(content, tuple):                      # older Gradio: (filepath, alt_text)
        content = [{"path": content[0]}]
    items = content if isinstance(content, list) else [content]
    parts = []
    for item in items:
        if item is None:
            continue
        if isinstance(item, str):
            if item.strip():
                parts.append(types.Part.from_text(text=item))
        elif isinstance(item, dict) and (item.get("type") == "text" or set(item) <= {"text", "type"}):
            if (item.get("text") or "").strip():
                parts.append(types.Part.from_text(text=item["text"]))
        elif isinstance(item, dict) and item.get("type") == "component":
            continue                                    # e.g. an image we generated earlier
        else:
            path = _path_of(item)
            if path and os.path.exists(path):
                parts.append(file_to_part(path))
    return parts


def history_to_contents(history):
    """Gradio chat history -> list of Gemini Content objects."""
    contents = []
    for msg in history or []:
        meta = msg.get("metadata") or {}
        if meta.get("title"):                           # skip "Thinking" / "Tool used" bubbles
            continue
        role = "user" if msg["role"] == "user" else "model"
        parts = content_to_parts(msg["content"])
        if not parts:
            continue
        if contents and contents[-1].role == role:      # Gemini prefers alternating turns
            contents[-1].parts.extend(parts)
        else:
            contents.append(types.Content(role=role, parts=parts))
    return contents


def message_to_parts(message):
    """The new user message: a string, or {"text": ..., "files": [...]} from a multimodal textbox."""
    if isinstance(message, dict):
        parts = [file_to_part(_path_of(f)) for f in message.get("files", []) if _path_of(f)]
        if (message.get("text") or "").strip():
            parts.append(types.Part.from_text(text=message["text"]))
        return parts
    return [types.Part.from_text(text=message)]


def build_contents(message, history):
    """Full conversation for Gemini = previous turns + the new user message."""
    contents = history_to_contents(history)
    new_parts = message_to_parts(message)
    if contents and contents[-1].role == "user":
        contents[-1].parts.extend(new_parts)
    else:
        contents.append(types.Content(role="user", parts=new_parts))
    return contents

print("Helpers ready.")

Helpers ready.


---
## Part 1 · `gr.Interface`: one function, one app

### The core idea

Every Gradio app is the same three things:

1. **A Python function**: your logic. It can be two lines or a call to an AI model.
2. **Input components**: how the user gives the function its arguments (textbox, slider, image, file...).
3. **Output components**: how the function's return values are displayed (text, markdown, JSON, table, image, audio...).

`gr.Interface` wires these together automatically. **One input component per function argument, one output component per return value, in the same order.** If you remember only one rule from this notebook, remember that one.

### Which Gradio class should I use?

| Class | Use it when | Effort |
|---|---|---|
| `gr.Interface` | One function, inputs on the left, outputs on the right | Lowest |
| `gr.ChatInterface` | A chatbot: your function takes `(message, history)` | Low |
| `gr.Blocks` | Custom layouts, several steps, tabs, buttons that trigger different functions | Most control |

### 1.1 Hello Gradio (no AI yet)

Start without any model so you can see Gradio on its own. Two inputs (a textbox and a slider) map to the two function arguments; one output displays the return value.

**Try:** change `"text"` to `gr.Textbox(lines=3)` or add a third input, and re-run.

In [6]:
def greet(name, excitement):
    return f"Hello, {name}" + "!" * int(excitement)

demo = gr.Interface(
    fn=greet,
    inputs=[gr.Textbox(label="Your name"), gr.Slider(1, 10, value=3, step=1, label="Excitement")],
    outputs=gr.Textbox(label="Greeting"),
    title="Hello Gradio",
    description="One function, two inputs, one output.",
    examples=[["Cohort 12", 5], ["Mohammad", 2]],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://074a671b1438533d86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**What just happened**
- `demo.launch()` started a small web server and displayed the app inside the notebook.
- `examples=` adds clickable sample inputs. Always add examples: they show users what your app is for.
- Add `share=True` to `launch()` to get a public link that works for 1 week, perfect for showing a demo to the cohort.
- Add `debug=True` while developing so errors print in the notebook.

### 1.2 AI Summariser: your first Gemini app

Now the function calls Gemini. Notice the pattern you will reuse everywhere:

1. Build a **prompt** from the user's inputs.
2. Call `client.models.generate_content(...)`.
3. Return `response.text`.

We use `gr.Markdown` as the output so Gemini's headings and bullet points render nicely, and `gr.Radio` / `gr.Dropdown` to turn prompt options into clickable controls. Turning prompt options into UI controls is the single most useful habit in AI app design.

In [7]:
def summarise(text, length, audience):
    if not text.strip():
        return "Please paste some text first."
    prompt = (
        f"Summarise the text below for {audience}.\n"
        f"Length: {length}.\n"
        "Use a one-line headline, then bullet points. Keep the original facts and numbers.\n\n"
        f"TEXT:\n{text}"
    )
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

SAMPLE = (
    "Decoding Data Science runs the AI Residency, a cohort-based programme where participants "
    "learn application architecture, build Gradio interfaces, version their work with Git and "
    "GitHub, and deploy AI applications to Hugging Face Spaces. Each cohort ends with residents "
    "presenting a working AI application to the community."
)

demo = gr.Interface(
    fn=summarise,
    inputs=[
        gr.Textbox(lines=10, label="Text to summarise", placeholder="Paste an article, email or report..."),
        gr.Radio(["3 bullet points", "1 short paragraph", "detailed (8-10 bullets)"],
                 value="3 bullet points", label="Length"),
        gr.Dropdown(["a busy executive", "a 12-year-old", "a technical team", "a LinkedIn audience"],
                    value="a busy executive", label="Audience"),
    ],
    outputs=gr.Markdown(label="Summary"),
    title="AI Summariser",
    examples=[[SAMPLE, "3 bullet points", "a busy executive"]],
    flagging_mode="never",
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f813de97cb199dab6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 1.3 Structured Extractor: Gemini returns JSON, Gradio shows a table

Chat answers are great for people. Apps and databases need **structured data**. Gemini can be forced to return JSON that matches a schema you define with **Pydantic**.

This app turns messy meeting notes into a clean action-item table. It shows two new output components:
- `gr.JSON`: the raw structured output (what your code or database receives).
- `gr.Dataframe`: the same data as a table (what a manager wants to see).

A function with two return values needs two output components.

In [ ]:
from pydantic import BaseModel

class ActionItem(BaseModel):
    task: str
    owner: str
    due_date: str
    priority: str          # High / Medium / Low

class MeetingSummary(BaseModel):
    meeting_title: str
    decisions: list[str]
    action_items: list[ActionItem]

def extract_actions(notes):
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Extract the meeting summary from these notes. Use 'Unassigned' or 'Not set' when missing.\n\n{notes}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=MeetingSummary,
        ),
    )
    data = response.parsed                              # a MeetingSummary object, already validated
    table = pd.DataFrame([a.model_dump() for a in data.action_items])
    return data.model_dump(), table

NOTES = """Cohort 12 planning call. Agreed MC03 will cover API calls with Postman, Python and SDKs.
Priya to update the LMS links by Tuesday. Daniel to record the Gradio walkthrough video this week, high priority.
Residents must push their Gradio app to GitHub before MC03. Someone needs to check the Hugging Face Spaces quota."""

demo = gr.Interface(
    fn=extract_actions,
    inputs=gr.Textbox(lines=8, label="Meeting notes"),
    outputs=[gr.JSON(label="Structured output (JSON)"), gr.Dataframe(label="Action items")],
    title="Meeting Notes to Action Items",
    examples=[[NOTES]],
    flagging_mode="never",
)
demo.launch()

### 1.4 Vision App: ask questions about an image

Gemini is **multimodal**: it understands images, PDFs and audio as well as text. With `gr.Image(type="filepath")` Gradio hands your function the path of the uploaded image, and our `file_to_part` helper sends it to Gemini.

**Ideas to try:** upload a screenshot of an error message, a chart from a report, a handwritten note, or a photo of a whiteboard architecture diagram.

In [ ]:
def describe_image(image_path, question):
    if image_path is None:
        return "Please upload an image."
    question = question.strip() or "Describe this image in detail."
    response = client.models.generate_content(
        model=MODEL,
        contents=[file_to_part(image_path), question],
    )
    return response.text

demo = gr.Interface(
    fn=describe_image,
    inputs=[
        gr.Image(type="filepath", label="Upload an image"),
        gr.Textbox(label="Your question", placeholder="What does this chart show? What is the error here?"),
    ],
    outputs=gr.Markdown(label="Answer"),
    title="Vision Q&A",
    flagging_mode="never",
)
demo.launch()

---
## Part 2 · `gr.ChatInterface`: eight kinds of chatbot

`gr.ChatInterface` builds a complete chat UI around one function with this signature:

```python
def my_chatbot(message, history):
    ...
    return "reply"          # or yield partial replies to stream
```

- `message`: the new user message (a string, or `{"text": ..., "files": [...]}` when `multimodal=True`).
- `history`: the previous turns, as a list of `{"role": "user" | "assistant", "content": ...}` dictionaries.

LLMs have **no memory**. Every call you send the whole conversation again. That is what `build_contents(message, history)` does for us.

The eight chatbots below go from the simplest to a full tool-using agent. Each one adds exactly one new idea.

| # | Chatbot | New idea |
|---|---|---|
| 2.1 | Basic chatbot | `(message, history)` + conversation memory |
| 2.2 | Streaming chatbot | `yield` for word-by-word replies |
| 2.3 | Persona chatbot | System instructions + `additional_inputs` (settings panel) |
| 2.4 | Multimodal chatbot | Images and PDFs inside the chat |
| 2.5 | Web-grounded chatbot | Google Search tool + citations |
| 2.6 | HR agent (tool calling) | The model calls your Python functions |
| 2.7 | Thinking chatbot | Showing the model's reasoning in a collapsible panel |
| 2.8 | Chat with your document | Answering only from an uploaded file |

### 2.1 Basic chatbot

The smallest useful chatbot. Ask it something, then ask a follow-up like "explain that more simply". It remembers, because we send the history every time.

In [ ]:
def basic_chat(message, history):
    response = client.models.generate_content(
        model=MODEL,
        contents=build_contents(message, history),
    )
    return response.text

demo = gr.ChatInterface(
    fn=basic_chat,
    title="Basic Gemini Chatbot",
    examples=["What is Gradio in one paragraph?", "Give me 3 project ideas for a beginner AI builder"],
)
demo.launch()

### 2.2 Streaming chatbot

Users hate waiting for a long answer to appear all at once. Streaming shows the answer as it is generated, exactly like ChatGPT, Claude or Gemini.

Two changes from 2.1:
1. Call `generate_content_stream` instead of `generate_content`.
2. Use `yield` instead of `return`, and yield the **whole answer so far** each time (not just the new chunk).

In [ ]:
def streaming_chat(message, history):
    answer = ""
    for chunk in client.models.generate_content_stream(
        model=MODEL,
        contents=build_contents(message, history),
    ):
        if chunk.text:
            answer += chunk.text
            yield answer

demo = gr.ChatInterface(
    fn=streaming_chat,
    title="Streaming Chatbot",
    examples=["Write a 300-word story about an AI resident shipping their first app"],
)
demo.launch()

### 2.3 Persona chatbot with a settings panel

Two powerful ideas in one app:

- **System instructions** give the model a role, rules and a tone that apply to the whole conversation. Same model, completely different product.
- **`additional_inputs`** adds controls under the chat. Their values are passed to your function as extra arguments, after `message` and `history`, in the same order.

**Teaching point:** temperature controls randomness. Low (0.0-0.3) for factual, consistent answers; high (1.0+) for creative writing.

In [ ]:
PERSONAS = {
    "AI Tutor": "You are a patient AI tutor for working professionals. Explain with simple analogies, "
                "check understanding with one short question at the end, and never use jargon without defining it.",
    "Mock Interviewer": "You are a senior interviewer for a Data Scientist role. Ask one question at a time, "
                        "wait for the answer, then give brief feedback and a score out of 10 before the next question.",
    "Socratic Coach": "You never give answers directly. Guide the learner with questions until they reach the answer themselves.",
    "Code Reviewer": "You are a strict but kind senior Python reviewer. Point out bugs, security issues "
                     "(especially exposed API keys) and readability problems, then show the corrected code.",
    "Arabic-English Translator": "Translate everything the user writes between Arabic and English. "
                                 "Give the translation first, then one line on tone or cultural nuance.",
}

def persona_chat(message, history, persona, temperature, model):
    config = types.GenerateContentConfig(
        system_instruction=PERSONAS[persona],
        temperature=temperature,
    )
    answer = ""
    for chunk in client.models.generate_content_stream(
        model=model, contents=build_contents(message, history), config=config
    ):
        if chunk.text:
            answer += chunk.text
            yield answer

demo = gr.ChatInterface(
    fn=persona_chat,
    title="Persona Chatbot",
    description="Pick a persona in Settings, then start chatting. Clear the chat when you switch persona.",
    additional_inputs=[
        gr.Dropdown(list(PERSONAS), value="AI Tutor", label="Persona"),
        gr.Slider(0.0, 2.0, value=0.7, step=0.1, label="Temperature"),
        gr.Dropdown([MODEL, FAST_MODEL], value=MODEL, label="Model"),
    ],
    additional_inputs_accordion=gr.Accordion("Settings", open=True),
)
demo.launch()

### 2.4 Multimodal chatbot: images and PDFs in the conversation

Set `multimodal=True` and the textbox accepts files. The message now arrives as `{"text": ..., "files": [...]}`; our helpers already handle that, so the function is identical to 2.2.

**Try:** upload a PDF of a policy and ask "What are the three most important rules here?", then follow up without re-uploading. The file stays in the history.

In [ ]:
def multimodal_chat(message, history):
    answer = ""
    for chunk in client.models.generate_content_stream(
        model=MODEL, contents=build_contents(message, history)
    ):
        if chunk.text:
            answer += chunk.text
            yield answer

demo = gr.ChatInterface(
    fn=multimodal_chat,
    multimodal=True,
    textbox=gr.MultimodalTextbox(
        file_types=["image", ".pdf", ".txt", ".md", ".csv"],
        file_count="multiple",
        placeholder="Type a message or attach images / PDFs...",
    ),
    title="Multimodal Chatbot",
)
demo.launch()

### 2.5 Web-grounded chatbot with citations

A plain LLM only knows what it learned during training. Adding the **Google Search tool** lets Gemini search the web when it needs current information, and the response includes the sources it used.

This is the simplest form of **grounding**: making the model's answer traceable to real sources. We append the sources as clickable links, which is what responsible AI apps should always do.

In [ ]:
def grounded_chat(message, history):
    response = client.models.generate_content(
        model=MODEL,
        contents=build_contents(message, history),
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
        ),
    )
    answer = response.text or ""

    sources = []
    meta = response.candidates[0].grounding_metadata if response.candidates else None
    for chunk in (meta.grounding_chunks or []) if meta else []:
        if chunk.web and chunk.web.uri:
            sources.append(f"- [{chunk.web.title or chunk.web.uri}]({chunk.web.uri})")
    if sources:
        answer += "\n\n**Sources**\n" + "\n".join(dict.fromkeys(sources))
    return answer

demo = gr.ChatInterface(
    fn=grounded_chat,
    title="Web-Grounded Chatbot",
    description="Answers current questions using Google Search, with sources.",
    examples=["What are the latest Gemini models released this month?", "What AI events are happening in Dubai this quarter?"],
)
demo.launch()

### 2.6 Agent chatbot: Gemini calls your Python functions (tool calling)

This is the step from **chatbot** to **agent**. We give Gemini ordinary Python functions. When a question needs data or an action, Gemini decides which function to call and with which arguments; the SDK runs it and sends the result back; Gemini then writes the final answer.

How the Google Gen AI SDK makes this easy:
- Pass the Python functions in `tools=[...]`. The **type hints and docstring become the tool description** the model reads, so write them carefully.
- The SDK runs the calls automatically (automatic function calling) and records them in `response.automatic_function_calling_history`.

We show every tool call to the user as a collapsible "Tool used" bubble, using `gr.ChatMessage(metadata={"title": ...})`. Transparency builds trust: users can see where an answer came from.

The example is an HR leave assistant, the same pattern as DecoBot. Swap the in-memory dictionary for a database or an API and this becomes a real enterprise agent.

In [ ]:
LEAVE_DB = {
    "E001": {"name": "Aisha Khan", "annual": 18, "sick": 5},
    "E002": {"name": "Rahul Mehta", "annual": 7, "sick": 10},
    "E003": {"name": "Omar Haddad", "annual": 22, "sick": 3},
}
POLICY = {
    "annual": "Employees receive 30 annual leave days per year. Requests over 5 days need manager approval 2 weeks in advance.",
    "sick": "Up to 15 sick days per year. A medical certificate is required for absences longer than 2 days.",
    "carry over": "Up to 10 unused annual days can be carried into the next year and must be used by 31 March.",
}

def get_leave_balance(employee_id: str) -> dict:
    """Get the remaining annual and sick leave days for an employee.

    Args:
        employee_id: The employee ID, for example "E001".
    """
    emp = LEAVE_DB.get(employee_id.upper())
    return emp if emp else {"error": f"No employee found with ID {employee_id}"}

def get_leave_policy(topic: str) -> str:
    """Look up the company leave policy.

    Args:
        topic: One of "annual", "sick" or "carry over".
    """
    return POLICY.get(topic.lower(), "Policy topic not found. Available topics: " + ", ".join(POLICY))

def apply_for_leave(employee_id: str, leave_type: str, days: int) -> dict:
    """Submit a leave request and deduct the days from the employee's balance.

    Args:
        employee_id: The employee ID, for example "E002".
        leave_type: Either "annual" or "sick".
        days: Number of days requested.
    """
    emp = LEAVE_DB.get(employee_id.upper())
    if not emp:
        return {"status": "rejected", "reason": "Unknown employee"}
    if leave_type not in ("annual", "sick"):
        return {"status": "rejected", "reason": "Leave type must be annual or sick"}
    if emp[leave_type] < days:
        return {"status": "rejected", "reason": f"Only {emp[leave_type]} {leave_type} days left"}
    emp[leave_type] -= days
    return {"status": "approved", "remaining": emp[leave_type]}

HR_TOOLS = [get_leave_balance, get_leave_policy, apply_for_leave]

def hr_agent(message, history):
    response = client.models.generate_content(
        model=MODEL,
        contents=build_contents(message, history),
        config=types.GenerateContentConfig(
            system_instruction="You are HALA, a friendly HR leave assistant. Always use the tools for balances, "
                               "policies and requests. Ask for the employee ID if it is missing. Never invent numbers.",
            tools=HR_TOOLS,
        ),
    )

    messages = []                                   # one "Tool used" bubble per call
    for content in response.automatic_function_calling_history or []:
        for part in content.parts or []:
            if part.function_call:
                messages.append(gr.ChatMessage(
                    role="assistant",
                    content=f"Arguments: `{json.dumps(dict(part.function_call.args or {}))}`",
                    metadata={"title": f"Tool used: {part.function_call.name}"},
                ))
            elif part.function_response and messages:
                messages[-1].content += f"\n\nResult: `{json.dumps(part.function_response.response, default=str)}`"
    messages.append(gr.ChatMessage(role="assistant", content=response.text or ""))
    return messages

demo = gr.ChatInterface(
    fn=hr_agent,
    title="HALA: HR Leave Agent (tool calling)",
    description="Employees: E001, E002, E003. Watch the 'Tool used' panels to see the agent's actions.",
    examples=[
        "How many annual leave days does E002 have?",
        "What is the carry-over policy?",
        "Book 3 days of annual leave for E001, then tell me the new balance",
    ],
)
demo.launch()

**Discussion for the cohort:** `apply_for_leave` changes data. In a real system, what checks would you add before an agent is allowed to call a function that changes something? (Hint: confirmation step, permissions, audit log, limits.)

### 2.7 Thinking chatbot: show the model's reasoning

Modern Gemini models **think** before they answer. With `include_thoughts=True` the API returns a summary of that reasoning as separate parts marked `part.thought`.

We stream both, putting the reasoning in a collapsible "Thinking" panel (status `pending` while it is still thinking, `done` when the answer starts) and the answer below it. This is the same pattern you see in the Gemini, Claude and ChatGPT apps.

**Try:** a logic puzzle or a maths word problem, then open the Thinking panel.

In [ ]:
def thinking_chat(message, history):
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(include_thoughts=True),
    )
    thoughts, answer = "", ""
    for chunk in client.models.generate_content_stream(
        model=MODEL, contents=build_contents(message, history), config=config
    ):
        candidate = chunk.candidates[0] if chunk.candidates else None
        if not candidate or not candidate.content or not candidate.content.parts:
            continue
        for part in candidate.content.parts:
            if not part.text:
                continue
            if part.thought:
                thoughts += part.text
            else:
                answer += part.text

        shown = []
        if thoughts:
            shown.append(gr.ChatMessage(
                role="assistant", content=thoughts,
                metadata={"title": "Thinking", "status": "done" if answer else "pending"},
            ))
        if answer:
            shown.append(gr.ChatMessage(role="assistant", content=answer))
        if shown:
            yield shown

demo = gr.ChatInterface(
    fn=thinking_chat,
    title="Thinking Chatbot",
    examples=[
        "A bootcamp has 3 tracks. Each resident picks 2 different tracks. How many different combinations are there? Explain.",
        "Our app returns HTTP 429 sometimes and 401 other times. What are the likely causes and fixes?",
    ],
)
demo.launch()

### 2.8 Chat with your document

Upload a PDF or text file once (in the panel under the chat) and every answer is based on it. The system instruction tells Gemini to answer **only** from the document and to say so when the answer is not there. That one rule removes most hallucinations in business use.

**How this compares to RAG:** here we send the whole document with every question. Gemini's very large context window makes this work for documents of hundreds of pages and it is the fastest way to build a prototype. When you have thousands of documents, you switch to **RAG** (retrieval-augmented generation): split documents into chunks, store them in a vector database, and send only the relevant chunks. We build that later in the Residency.

In [ ]:
DOC_RULES = (
    "Answer only from the attached document. Quote or name the section you used. "
    "If the answer is not in the document, say: 'I could not find that in the document.'"
)

def document_chat(message, history, document):
    if document is None:
        yield "Please upload a PDF or text file in the 'Your document' panel below first."
        return
    contents = [
        types.Content(role="user", parts=[file_to_part(document),
                                          types.Part.from_text(text="This is the reference document for our conversation.")]),
        types.Content(role="model", parts=[types.Part.from_text(text="Understood. I will answer only from this document.")]),
    ] + build_contents(message, history)

    answer = ""
    for chunk in client.models.generate_content_stream(
        model=MODEL, contents=contents,
        config=types.GenerateContentConfig(system_instruction=DOC_RULES, temperature=0.2),
    ):
        if chunk.text:
            answer += chunk.text
            yield answer

demo = gr.ChatInterface(
    fn=document_chat,
    title="Chat With Your Document",
    additional_inputs=[gr.File(label="Your document (PDF, TXT, MD)", file_types=[".pdf", ".txt", ".md"])],
    additional_inputs_accordion=gr.Accordion("Your document", open=True),
    examples=[["Summarise this document in 5 bullet points", None], ["What deadlines are mentioned?", None]],
)
demo.launch()

---
## Part 3 · `gr.Blocks`: full control over layout and behaviour

`gr.Interface` and `gr.ChatInterface` decide the layout for you. `gr.Blocks` lets you decide everything:

```python
with gr.Blocks() as demo:
    with gr.Row():                       # side by side
        with gr.Column():                # stacked
            box = gr.Textbox()
            btn = gr.Button("Run")
        out = gr.Markdown()
    btn.click(fn=my_function, inputs=box, outputs=out)   # the event
```

Three things to learn:
1. **Layout**: `gr.Row`, `gr.Column`, `gr.Tab`, `gr.Accordion`, `gr.Sidebar`.
2. **Events**: `button.click`, `textbox.submit`, `component.change`, `chatbot.like`, and `.then(...)` to chain steps.
3. **State and updates**: components can be inputs and outputs at the same time, and a function can return `gr.update(...)` to change a component (for example, new dropdown choices).

### 3.1 Model Arena: compare two models side by side

One prompt, two models, shown next to each other with response times. Use this in class to discuss the **speed / cost / quality trade-off**: when is the cheaper Flash-Lite model good enough?

New concepts: `gr.Row` / `gr.Column`, a button `click` event with several outputs, and `gr.Examples` inside Blocks.

In [ ]:
def run_model(prompt, model):
    start = time.time()
    response = client.models.generate_content(model=model, contents=prompt)
    return response.text, f"{model} · {time.time() - start:.1f} s"

def arena(prompt, model_a, model_b):
    text_a, info_a = run_model(prompt, model_a)
    text_b, info_b = run_model(prompt, model_b)
    return text_a, info_a, text_b, info_b

MODELS = [MODEL, FAST_MODEL]

with gr.Blocks(title="Model Arena") as demo:
    gr.Markdown("# Model Arena\nSend one prompt to two models and compare quality and speed.")
    prompt = gr.Textbox(lines=3, label="Prompt")
    run_btn = gr.Button("Compare", variant="primary")
    with gr.Row():
        with gr.Column():
            model_a = gr.Dropdown(MODELS, value=MODEL, label="Model A")
            info_a = gr.Markdown()
            out_a = gr.Markdown()
        with gr.Column():
            model_b = gr.Dropdown(MODELS, value=FAST_MODEL, label="Model B")
            info_b = gr.Markdown()
            out_b = gr.Markdown()
    gr.Examples(
        [["Explain the difference between an API and a web UI to a non-technical manager."],
         ["Write a Python function that validates an email address, with tests."]],
        inputs=prompt,
    )
    run_btn.click(arena, inputs=[prompt, model_a, model_b], outputs=[out_a, info_a, out_b, info_b])

demo.launch()

### 3.2 Custom chatbot built from scratch, with feedback and export

`gr.ChatInterface` is quick, but production apps often need more: extra buttons, logging, exports, side panels. Here we build a chatbot from individual Blocks components:

- `gr.Chatbot` displays the conversation. Its value **is** the history list.
- `textbox.submit` and `button.click` both trigger the same streaming function.
- `chatbot.like` captures thumbs up / down feedback. This is how real teams collect evaluation data to improve prompts.
- `gr.State` keeps the feedback log for this user's session.
- An export button writes the transcript to a Markdown file the user can download.

In [ ]:
def respond(message, history):
    if not message.strip():
        yield "", history
        return
    history = list(history or []) + [{"role": "user", "content": message}]
    contents = history_to_contents(history)
    answer = ""
    yield "", history + [{"role": "assistant", "content": ""}]
    for chunk in client.models.generate_content_stream(model=MODEL, contents=contents):
        if chunk.text:
            answer += chunk.text
            yield "", history + [{"role": "assistant", "content": answer}]

def record_feedback(feedback_log, evt: gr.LikeData):
    feedback_log = list(feedback_log or [])
    feedback_log.append({"message_index": evt.index, "liked": evt.liked,
                         "message": str(evt.value)[:200], "time": time.strftime("%H:%M:%S")})
    return feedback_log, feedback_log

def export_transcript(history):
    lines = ["# Chat transcript\n"]
    for msg in history or []:
        text = " ".join(p.text for p in content_to_parts(msg["content"]) if p.text)
        lines.append(f"**{msg['role'].title()}:** {text}\n")
    path = "chat_transcript.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path

with gr.Blocks(title="DDS Assistant") as demo:
    gr.Markdown("# DDS Assistant\nA chatbot built from individual components.")
    feedback_state = gr.State([])
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=450, label="Conversation")
            with gr.Row():
                msg = gr.Textbox(placeholder="Ask anything and press Enter...", show_label=False, scale=4)
                send = gr.Button("Send", variant="primary", scale=1)
            with gr.Row():
                gr.ClearButton([msg, chatbot], value="New chat")
                export_btn = gr.Button("Export transcript")
        with gr.Column(scale=2):
            gr.Markdown("### Feedback log\nRate answers with the thumbs on each reply.")
            feedback_view = gr.JSON(label="Feedback")
            transcript_file = gr.File(label="Transcript download")

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send.click(respond, [msg, chatbot], [msg, chatbot])
    chatbot.like(record_feedback, [feedback_state], [feedback_state, feedback_view])
    export_btn.click(export_transcript, chatbot, transcript_file)

demo.launch()

### 3.3 AI Data Analyst: upload a CSV, see it, chart it, ask questions about it

A complete mini-product, and a template for many business apps:

1. **Upload** a CSV (or click *Load sample data*).
2. The app shows a **preview table** and fills the chart dropdowns with the file's column names using `gr.update(choices=...)`. Components that react to data like this are what make an app feel professional.
3. A native **bar chart** (`gr.BarPlot`) updates when you pick columns.
4. **Ask a question** in plain English. We send Gemini the column types, summary statistics and a sample of rows, never the API key and not necessarily the whole file.

**Teaching point:** for large files, the reliable pattern is to let the model write pandas code (or call a pandas tool) instead of reading all the rows. That is the agent pattern from 2.6 applied to data.

In [ ]:
def sample_data():
    rng = np.random.default_rng(12)
    months = pd.date_range("2026-01-01", periods=9, freq="MS").strftime("%b %Y")
    rows = []
    for region in ["UAE", "KSA", "UK", "India"]:
        for m in months:
            rows.append({"month": m, "region": region,
                         "enrolments": int(rng.integers(20, 120)),
                         "revenue_usd": int(rng.integers(2000, 15000))})
    path = "sample_enrolments.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def load_csv(file_path):
    df = pd.read_csv(file_path)
    cols = list(df.columns)
    numeric = list(df.select_dtypes("number").columns) or cols
    return (df, df.head(20),
            gr.update(choices=cols, value=cols[0]),
            gr.update(choices=numeric, value=numeric[0]),
            f"Loaded **{len(df):,} rows** and **{len(cols)} columns**.")

def make_chart(df, x_col, y_col):
    if df is None or x_col is None or y_col is None:
        return None
    return df.groupby(x_col, as_index=False)[y_col].sum()

def ask_data(df, question):
    if df is None:
        return "Load a CSV first."
    context = (
        f"Columns and types:\n{df.dtypes.to_string()}\n\n"
        f"Summary statistics:\n{df.describe(include='all').to_string()}\n\n"
        f"First 30 rows (CSV):\n{df.head(30).to_csv(index=False)}"
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=f"You are a data analyst. Use only the data below. Show the numbers you used.\n\n{context}\n\nQuestion: {question}",
    )
    return response.text

with gr.Blocks(title="AI Data Analyst") as demo:
    gr.Markdown("# AI Data Analyst")
    df_state = gr.State(None)
    with gr.Row():
        csv_file = gr.File(label="Upload CSV", file_types=[".csv"], scale=3)
        sample_btn = gr.Button("Load sample data", scale=1)
    status = gr.Markdown()
    preview = gr.Dataframe(label="Preview (first 20 rows)", interactive=False)
    with gr.Row():
        x_col = gr.Dropdown(label="Group by (x axis)")
        y_col = gr.Dropdown(label="Sum of (y axis)")
    chart = gr.BarPlot(x=None, y=None, label="Chart")
    question = gr.Textbox(label="Ask a question about the data",
                          placeholder="Which region had the highest revenue? Any trend over time?")
    ask_btn = gr.Button("Ask Gemini", variant="primary")
    answer = gr.Markdown()

    load_outputs = [df_state, preview, x_col, y_col, status]
    csv_file.upload(load_csv, csv_file, load_outputs)
    sample_btn.click(sample_data, None, csv_file).then(load_csv, csv_file, load_outputs)

    def update_chart(df, x, y):
        data = make_chart(df, x, y)
        return gr.BarPlot(value=data, x=x, y=y) if data is not None else gr.BarPlot()
    for comp in (x_col, y_col):
        comp.change(update_chart, [df_state, x_col, y_col], chart)

    ask_btn.click(ask_data, [df_state, question], answer)
    question.submit(ask_data, [df_state, question], answer)

demo.launch()

---
## Part 4 · Voice, audio and images

Gradio has components for every media type, and Gemini can both understand and generate media. This part shows the three most requested demos.

| Component | Gives your function | Your function returns |
|---|---|---|
| `gr.Image(type="filepath")` | path to the image | path, PIL image or numpy array |
| `gr.Audio(type="filepath")` | path to the recording | path, or `(sample_rate, numpy_array)` |
| `gr.Video` | path to the video | path to a video |
| `gr.File` | path to the file | path to a file to download |

### 4.1 Text-to-speech

Gemini's TTS model returns raw audio (16-bit PCM at 24,000 Hz). Gradio plays it directly if we return `(sample_rate, numpy_array)`.

**Try:** add style instructions in the text itself, for example *"Say cheerfully: Welcome to Cohort 12!"*

In [ ]:
VOICES = ["Kore", "Puck", "Charon", "Zephyr", "Fenrir", "Aoede"]

def text_to_speech(text, voice):
    response = client.models.generate_content(
        model=TTS_MODEL,
        contents=text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)
                )
            ),
        ),
    )
    pcm = response.candidates[0].content.parts[0].inline_data.data
    return 24000, np.frombuffer(pcm, dtype=np.int16)

demo = gr.Interface(
    fn=text_to_speech,
    inputs=[gr.Textbox(lines=4, label="Text", value="Say warmly: Welcome to the AI Residency. Today, we build and ship."),
            gr.Dropdown(VOICES, value="Kore", label="Voice")],
    outputs=gr.Audio(label="Speech", autoplay=True),
    title="Text to Speech",
    flagging_mode="never",
)
demo.launch()

### 4.2 Voice assistant: speak a question, hear the answer

This chains everything: microphone input, Gemini understanding the audio directly (no separate speech-to-text step), a text answer, and the answer spoken back with the function from 4.1.

New concept: a function with **two outputs** that feed two different components, and `.then()` would be the alternative if you wanted the text to appear before the audio is ready.

In [ ]:
def voice_assistant(audio_path, voice):
    if audio_path is None:
        return "Record or upload a question first.", None
    response = client.models.generate_content(
        model=MODEL,
        contents=[file_to_part(audio_path),
                  "Listen to this question and answer it clearly in under 80 words, as if speaking aloud."],
    )
    answer = response.text
    return answer, text_to_speech(answer, voice)

demo = gr.Interface(
    fn=voice_assistant,
    inputs=[gr.Audio(sources=["microphone", "upload"], type="filepath", label="Ask by voice"),
            gr.Dropdown(VOICES, value="Puck", label="Reply voice")],
    outputs=[gr.Markdown(label="Answer"), gr.Audio(label="Spoken answer", autoplay=True)],
    title="Voice Assistant",
    flagging_mode="never",
)
demo.launch()

### 4.3 Image Studio: generate and edit images

The image model (Nano Banana) creates images from text, and **edits** images when you also send one: "make the background a Dubai skyline", "turn this sketch into a clean diagram".

The response can contain text parts and image parts; we pick out both. The SDK accepts PIL images directly in `contents`.

In [ ]:
def image_studio(prompt, reference_image):
    if not prompt.strip():
        return None, "Write a prompt first."
    contents = [prompt]
    if reference_image is not None:
        contents.append(Image.open(reference_image))
    response = client.models.generate_content(
        model=IMAGE_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]),
    )
    image, notes = None, ""
    for part in response.candidates[0].content.parts:
        if part.inline_data and part.inline_data.data:
            image = Image.open(io.BytesIO(part.inline_data.data))
        elif part.text:
            notes += part.text
    return image, notes or "Done."

demo = gr.Interface(
    fn=image_studio,
    inputs=[gr.Textbox(lines=3, label="Prompt",
                       placeholder="A clean flat illustration of a client-server-model architecture, navy and cyan"),
            gr.Image(type="filepath", label="Optional image to edit")],
    outputs=[gr.Image(label="Result"), gr.Markdown(label="Model notes")],
    title="Image Studio",
    flagging_mode="never",
)
demo.launch()

---
## Part 5 · Capstone: the DDS AI Playground

Everything comes together in one app:

- `gr.Sidebar` holds **global settings** (model, temperature) that every tab uses.
- `gr.Tabs` organise six tools in one place.
- A `gr.ChatInterface` is **rendered inside** a Blocks tab, and its `additional_inputs` point to the sidebar controls.
- `gr.themes.Soft()` changes the whole look with one argument.

It reuses the functions from Parts 1-4, so **run those cells first**. This is the structure to copy for your own Residency project: one app, several tabs, one settings panel.

In [ ]:
def playground_chat(message, history, model, temperature):
    answer = ""
    for chunk in client.models.generate_content_stream(
        model=model, contents=build_contents(message, history),
        config=types.GenerateContentConfig(temperature=temperature),
    ):
        if chunk.text:
            answer += chunk.text
            yield answer

with gr.Blocks(title="DDS AI Playground", theme=gr.themes.Soft()) as playground:
    with gr.Sidebar():
        gr.Markdown("## Settings")
        pg_model = gr.Dropdown([MODEL, FAST_MODEL], value=MODEL, label="Model")
        pg_temp = gr.Slider(0.0, 2.0, value=0.7, step=0.1, label="Temperature")
        gr.Markdown("Decoding Data Science · AI Residency")

    gr.Markdown("# DDS AI Playground\nSix AI tools built with Gradio and Gemini.")

    with gr.Tabs():
        with gr.Tab("Chat"):
            gr.ChatInterface(fn=playground_chat, multimodal=True,
                             additional_inputs=[pg_model, pg_temp])

        with gr.Tab("Summarise"):
            s_text = gr.Textbox(lines=8, label="Text")
            s_len = gr.Radio(["3 bullet points", "1 short paragraph"], value="3 bullet points", label="Length")
            s_aud = gr.Dropdown(["a busy executive", "a technical team"], value="a busy executive", label="Audience")
            s_btn = gr.Button("Summarise", variant="primary")
            s_out = gr.Markdown()
            s_btn.click(summarise, [s_text, s_len, s_aud], s_out)

        with gr.Tab("Vision"):
            with gr.Row():
                v_img = gr.Image(type="filepath", label="Image")
                with gr.Column():
                    v_q = gr.Textbox(label="Question")
                    v_btn = gr.Button("Ask", variant="primary")
                    v_out = gr.Markdown()
            v_btn.click(describe_image, [v_img, v_q], v_out)

        with gr.Tab("HR Agent"):
            gr.ChatInterface(fn=hr_agent, examples=["What is the sick leave policy?",
                                                    "How many annual days does E003 have?"])

        with gr.Tab("Image Studio"):
            with gr.Row():
                with gr.Column():
                    i_prompt = gr.Textbox(lines=3, label="Prompt")
                    i_ref = gr.Image(type="filepath", label="Optional image to edit")
                    i_btn = gr.Button("Generate", variant="primary")
                with gr.Column():
                    i_out = gr.Image(label="Result")
                    i_notes = gr.Markdown()
            i_btn.click(image_studio, [i_prompt, i_ref], [i_out, i_notes])

        with gr.Tab("Voice"):
            with gr.Row():
                with gr.Column():
                    a_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Ask by voice")
                    a_voice = gr.Dropdown(VOICES, value="Puck", label="Reply voice")
                    a_btn = gr.Button("Answer", variant="primary")
                with gr.Column():
                    a_text = gr.Markdown()
                    a_audio = gr.Audio(label="Spoken answer", autoplay=True)
            a_btn.click(voice_assistant, [a_in, a_voice], [a_text, a_audio])

playground.launch()

---
## Part 6 · Share and deploy

### 6.1 Sharing options from Colab

| Option | Code | Use it for |
|---|---|---|
| Inline in the notebook | `demo.launch()` | Building and testing |
| Public link (1 week) | `demo.launch(share=True)` | Showing a demo to the cohort or a client today |
| Password protected | `demo.launch(share=True, auth=("cohort12", "choose-a-password"))` | Anything with real data or a paid API key behind it |
| See errors while developing | `demo.launch(debug=True)` | Debugging |

Remember: anyone with your share link is **using your API key**. Add `auth=` and keep links short-lived.

If you see "port already in use" or old apps misbehave, run `gr.close_all()`.

In [ ]:
gr.close_all()   # stop every running Gradio app in this notebook

### 6.2 Deploy permanently to Hugging Face Spaces

A Space is a free, permanent home for your app with a public URL, perfect for your GitHub README and LinkedIn.

A Gradio Space needs two files:
- `app.py`: your app code, ending with `demo.launch()`.
- `requirements.txt`: the libraries to install.

**Steps**
1. Create a Space at https://huggingface.co/new-space and choose the **Gradio** SDK.
2. In the Space's **Settings → Variables and secrets**, add a secret named `GEMINI_API_KEY`. This is the Spaces equivalent of Colab Secrets.
3. Upload `app.py` and `requirements.txt` (the cell below creates them), or push them with Git:
   `git add . && git commit -m "First version of my Gemini chatbot" && git push`
4. The Space builds and gives you a live URL. Share it in the cohort WhatsApp group.

The cell below writes a complete, deployable streaming chatbot. Notice it reads the key from `os.environ`, never from the code.

In [ ]:
%%writefile app.py
import os
import gradio as gr
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])   # set in Space secrets
MODEL = "gemini-3.8-flash"

def to_contents(message, history):
    contents = []
    for msg in history:
        content = msg["content"]
        items = content if isinstance(content, list) else [content]
        text = " ".join(i if isinstance(i, str) else i.get("text", "") for i in items if isinstance(i, (str, dict)))
        if text.strip():
            role = "user" if msg["role"] == "user" else "model"
            contents.append(types.Content(role=role, parts=[types.Part.from_text(text=text)]))
    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=message)]))
    return contents

def chat(message, history):
    answer = ""
    for chunk in client.models.generate_content_stream(model=MODEL, contents=to_contents(message, history)):
        if chunk.text:
            answer += chunk.text
            yield answer

demo = gr.ChatInterface(chat, title="My Gemini Chatbot", description="Built in the DDS AI Residency")

if __name__ == "__main__":
    demo.launch()

In [ ]:
%%writefile requirements.txt
gradio
google-genai

### 6.3 Turn your app into an MCP server (tools for AI agents)

**MCP (Model Context Protocol)** is the standard way AI assistants such as Claude, Cursor and many agent frameworks connect to external tools. Gradio can expose any app as an MCP server with one argument; each function becomes a tool, and its **docstring and type hints become the tool description**.

This means the HR tools you wrote in 2.6 can be used by other AI agents, not just by people.

In [ ]:
!pip install -q "gradio[mcp]"

In [ ]:
def leave_balance_tool(employee_id: str) -> str:
    """Get the remaining annual and sick leave days for an employee ID such as E001."""
    return json.dumps(get_leave_balance(employee_id))

mcp_demo = gr.Interface(fn=leave_balance_tool, inputs=gr.Textbox(label="Employee ID"),
                        outputs=gr.Textbox(label="Balance"), title="Leave Balance Tool")
mcp_demo.launch(mcp_server=True)   # the MCP endpoint URL is printed in the output and shown in the app footer

### 6.4 Call any Gradio app from Python

Every Gradio app automatically has an API. The **"Use via API"** link in the app footer shows the exact code. With `gradio_client` you can call your own Space (or any public Space) from Python, which is how you connect a Gradio prototype to other systems.

```python
from gradio_client import Client

app = Client("your-username/your-space-name")
result = app.predict("What is Gradio?", api_name="/chat")
print(result)
```

---
## Part 7 · Reference

### 7.1 Component cheat sheet

| Need | Component | Notes |
|---|---|---|
| Short or long text | `gr.Textbox(lines=...)` | `placeholder=`, `label=` |
| Formatted AI output | `gr.Markdown()` | Renders headings, lists, tables, code |
| Choose one option | `gr.Radio`, `gr.Dropdown` | `choices=`, `value=` |
| Choose several | `gr.CheckboxGroup`, `gr.Dropdown(multiselect=True)` | |
| Number | `gr.Slider`, `gr.Number` | `minimum`, `maximum`, `step` |
| Image in / out | `gr.Image(type="filepath" / "pil")` | |
| Audio in / out | `gr.Audio(sources=["microphone"])` | Return `(rate, array)` to play |
| Files | `gr.File`, `gr.UploadButton`, `gr.DownloadButton` | `file_types=[".pdf"]` |
| Tables | `gr.Dataframe` | Pass a pandas DataFrame |
| Charts | `gr.BarPlot`, `gr.LinePlot`, `gr.ScatterPlot`, `gr.Plot` | `gr.Plot` accepts matplotlib / plotly |
| Structured data | `gr.JSON` | |
| Chat | `gr.Chatbot`, `gr.ChatInterface`, `gr.MultimodalTextbox` | |
| Layout | `gr.Row`, `gr.Column`, `gr.Tab`, `gr.Accordion`, `gr.Sidebar` | Only inside `gr.Blocks` |
| Memory per user | `gr.State` | Not shared between users |

### 7.2 Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `SecretNotFoundError` or key is `None` | Secret missing or notebook access off | Key icon → check name `GEMINI_API_KEY` → Notebook access on |
| `400 ... API key not valid` / `401` | Wrong or revoked key | Create a new key in AI Studio, update the secret |
| `404 ... model not found` | Model renamed or not available to your key | Run the model list cell in 0.3, update `MODEL` |
| `429 RESOURCE_EXHAUSTED` | Free-tier rate limit | Wait a minute, use `FAST_MODEL`, or enable billing |
| App shows "Error" with no detail | Exception inside your function | `demo.launch(debug=True)` and read the traceback |
| Old app still running / port in use | Several launched apps | `gr.close_all()` |
| Chatbot forgets earlier messages | History not sent to the model | Use `build_contents(message, history)` |
| Space build fails | Missing package or secret | Check the **Logs** tab, `requirements.txt` and Space secrets |

### 7.3 Exercises for residents

1. **Interface:** Extend the summariser (1.2) with a "Language" dropdown (English, Arabic, Hindi, French).
2. **Structured output:** Change the extractor (1.3) to read a CV and return name, skills, years of experience and a fit score for a job description.
3. **Persona:** Add your own persona to 2.3 for your job role, and test it with three realistic questions.
4. **Agent:** Add a tool to 2.6, for example `cancel_leave(employee_id, days)`, and make the agent ask for confirmation before calling it.
5. **Blocks:** Add a "Download chart data" button to the Data Analyst (3.3).
6. **Ship it:** Pick your favourite app, deploy it to Hugging Face Spaces with the key in Secrets, push the code to GitHub with a README, and share both links in the cohort group.

### 7.4 Further reading
- Gradio guides: https://www.gradio.app/guides
- Gradio chatbot guides: https://www.gradio.app/guides/creating-a-chatbot-fast
- Gemini API documentation: https://ai.google.dev/gemini-api/docs
- Google Gen AI Python SDK: https://googleapis.github.io/python-genai/

---
*Decoding Data Science · AI Residency · Build smarter. Collaborate better. Ship with confidence.*